# NB15 — Gradient Descent (from scratch)

> **The alternative to the normal equations — essential for large datasets and neural networks.**

Normal equations: solve `(XᵀX)β = Xᵀy` exactly in one step.
Gradient descent: take many small steps in the direction of steepest descent.

For millions of parameters or rows, gradient descent wins.


## The gradient of SSR

```
SSR = Σ(yᵢ − β₀ − β₁xᵢ)²

∂SSR/∂β₀ = -2 Σ(yᵢ − ŷᵢ)
∂SSR/∂β₁ = -2 Σ xᵢ(yᵢ − ŷᵢ)

Update rule:
  β₀ ← β₀ − α · ∂SSR/∂β₀ / n
  β₁ ← β₁ − α · ∂SSR/∂β₁ / n
```

α is the **learning rate** — controls step size.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
X = np.array([1.,2.,3.,4.,5.,6.,7.,8.,9.,10.])
y = np.array([40.,45.,50.,55.,65.,70.,75.,85.,90.,95.])
n = len(X)

def ols_gradient_descent(X, y, lr=0.001, epochs=5000):
    b0, b1 = 0.0, 0.0
    history = []

    for epoch in range(epochs):
        y_hat = b0 + b1 * X
        resid  = y - y_hat
        ssr    = np.sum(resid**2)

        # Gradients
        grad_b0 = -2 * np.sum(resid) / n
        grad_b1 = -2 * np.sum(X * resid) / n

        # Update
        b0 -= lr * grad_b0
        b1 -= lr * grad_b1

        history.append({'epoch': epoch, 'ssr': ssr, 'b0': b0, 'b1': b1})

    return b0, b1, history

b0_gd, b1_gd, hist = ols_gradient_descent(X, y, lr=0.001, epochs=5000)
print(f"Gradient Descent: β₀={b0_gd:.4f}, β₁={b1_gd:.4f}")

# Compare with normal equations
xbar, ybar = X.mean(), y.mean()
b1_ols = np.sum((X-xbar)*(y-ybar)) / np.sum((X-xbar)**2)
b0_ols = ybar - b1_ols * xbar
print(f"Normal Equations: β₀={b0_ols:.4f}, β₁={b1_ols:.4f}")
print(f"Match: {np.allclose([b0_gd, b1_gd], [b0_ols, b1_ols], atol=0.01)}")

# Plot SSR convergence
ssrs = [h['ssr'] for h in hist]
plt.figure(figsize=(8,4))
plt.plot(ssrs, color='steelblue', linewidth=1.5)
plt.xlabel('Epoch'); plt.ylabel('SSR')
plt.title('Gradient Descent: SSR decreases each epoch')
plt.yscale('log'); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


In [ ]:
# Visualise the descent on the SSR surface
import numpy as np
import matplotlib.pyplot as plt

# SSR surface over (b0, b1) grid
b0_vals = np.linspace(10, 60, 100)
b1_vals = np.linspace(3, 8, 100)
B0, B1  = np.meshgrid(b0_vals, b1_vals)
SSR = np.sum((y - (B0[:,:,None] + B1[:,:,None]*X[None,None,:]))**2, axis=2)

b0_gd2, b1_gd2, hist2 = ols_gradient_descent(X, y, lr=0.0005, epochs=3000)

path_b0 = [h['b0'] for h in hist2[::30]]
path_b1 = [h['b1'] for h in hist2[::30]]

fig, ax = plt.subplots(figsize=(8, 6))
cp = ax.contour(B0, B1, np.log(SSR+1), levels=25, cmap='viridis')
ax.plot(path_b0, path_b1, 'r.-', markersize=6, linewidth=1.5, label='GD path')
ax.plot(b0_ols, b1_ols, 'w*', markersize=15, label='OLS minimum')
ax.set_xlabel('β₀ (intercept)'); ax.set_ylabel('β₁ (slope)')
ax.set_title('Gradient Descent on SSR surface')
ax.legend(); plt.colorbar(cp, label='log(SSR+1)')
plt.tight_layout(); plt.show()


In [ ]:
# Mini-batch Stochastic Gradient Descent (SGD)
import numpy as np

def sgd(X, y, lr=0.01, epochs=100, batch_size=4, seed=0):
    np.random.seed(seed)
    b0, b1 = 0.0, 0.0
    n = len(X)
    history = []
    for epoch in range(epochs):
        idx = np.random.permutation(n)
        X_s, y_s = X[idx], y[idx]
        for i in range(0, n, batch_size):
            Xb = X_s[i:i+batch_size]
            yb = y_s[i:i+batch_size]
            resid     = yb - (b0 + b1*Xb)
            b0 += lr * 2*resid.mean()
            b1 += lr * 2*(Xb*resid).mean()
        y_hat = b0 + b1*X
        history.append(np.sum((y-y_hat)**2))
    return b0, b1, history

b0_s, b1_s, hist_s = sgd(X, y, lr=0.005, epochs=300, batch_size=3)
print(f"SGD result:  β₀={b0_s:.4f}, β₁={b1_s:.4f}")
print(f"OLS result:  β₀={b0_ols:.4f}, β₁={b1_ols:.4f}")

import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.plot(hist_s, color='orange', linewidth=1.5, label='SGD (batch=3)')
plt.axhline(np.sum((y-(b0_ols+b1_ols*X))**2), color='green',
            linestyle='--', linewidth=1.5, label='OLS minimum SSR')
plt.xlabel('Epoch'); plt.ylabel('SSR')
plt.title('Mini-batch SGD convergence')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


## Key Takeaways

| Method | When to use |
|--------|------------|
| Normal equations | Small/medium n, exact solution needed |
| Batch GD | Medium n, when normal equations too slow |
| Mini-batch SGD | Large n, foundation of deep learning |
| Learning rate | Too large → diverges. Too small → slow. Use adaptive (Adam, RMSProp) |

**Next:** NB16 — Robust regression (handling outliers).
